In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

MODEL_FILES = {
    "LightGBM":
        "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",

    "RF":
        "Prescriptive/RF_prescriptive_optimizer_inputs.csv",

    "XGBoost":
        "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv",
}

COHORT_PATH = "Cohort/fixed_cohort.csv"

DURATION_CAP = 360.0

LAMBDAS = np.round(
    np.arange(0.0, 1.01, 0.1),
    1
)

OUTPUT_DIR = "Results"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Duration-cap audit configuration loaded.")
print("Duration cap:", DURATION_CAP)
print("Lambda grid:", LAMBDAS)

In [ ]:
# ============================================================
# 2. LOAD FIXED 12-CASE COHORT
# ============================================================

cohort_df = pd.read_csv(
    COHORT_PATH
)

cohort_ids = cohort_df[
    "LOG_ID"
].tolist()

assert len(cohort_ids) == 12, (
    f"Expected 12 fixed cases, found {len(cohort_ids)}."
)

assert len(set(cohort_ids)) == 12, (
    "Duplicate LOG_ID detected in fixed cohort."
)

print("Fixed cohort loaded.")
print("Number of cases:", len(cohort_ids))

In [ ]:
# ============================================================
# 3. DURATION-CAP AUDIT FUNCTION
# ============================================================

def audit_duration_cap(
    df,
    model_name,
    cohort_name,
    duration_cap=360.0
):

    results = []

    p50 = (
        df["DURATION_P50_MINS"]
        .to_numpy(dtype=float)
    )

    p90 = (
        df["DURATION_P90_MINS"]
        .to_numpy(dtype=float)
    )

    for lam in LAMBDAS:

        duration_uncapped = (
            p50
            + lam * (p90 - p50)
        )

        duration_capped = np.minimum(
            duration_uncapped,
            duration_cap
        )

        cap_mask = (
            duration_uncapped > duration_cap
        )

        n_capped = int(
            cap_mask.sum()
        )

        pct_capped = float(
            cap_mask.mean() * 100
        )

        # Total number of minutes removed by the cap
        total_reduction = float(
            np.sum(
                duration_uncapped
                - duration_capped
            )
        )

        # Mean reduction across all cases
        mean_reduction_all = float(
            np.mean(
                duration_uncapped
                - duration_capped
            )
        )

        # Mean reduction only among affected cases
        if n_capped > 0:

            mean_reduction_capped = float(
                np.mean(
                    duration_uncapped[cap_mask]
                    - duration_capped[cap_mask]
                )
            )

            max_reduction = float(
                np.max(
                    duration_uncapped[cap_mask]
                    - duration_capped[cap_mask]
                )
            )

        else:

            mean_reduction_capped = 0.0
            max_reduction = 0.0

        results.append({

            "Cohort":
                cohort_name,

            "Model":
                model_name,

            "Lambda":
                lam,

            "N":
                len(df),

            "Duration_Cap":
                duration_cap,

            "N_Capped":
                n_capped,

            "Pct_Capped":
                pct_capped,

            "Max_Uncapped":
                float(
                    np.max(duration_uncapped)
                ),

            "Max_Capped":
                float(
                    np.max(duration_capped)
                ),

            "Total_Minutes_Removed":
                total_reduction,

            "Mean_Reduction_All_Cases":
                mean_reduction_all,

            "Mean_Reduction_Capped_Cases":
                mean_reduction_capped,

            "Max_Reduction":
                max_reduction
        })

    return pd.DataFrame(results)

In [ ]:
# ============================================================
# 4. FULL TEST-COHORT CAP AUDIT
# ============================================================

full_audit_results = []

for model_name, path in MODEL_FILES.items():

    print(
        f"Auditing full test cohort: {model_name}"
    )

    df = pd.read_csv(
        path
    )

    assert len(df) == 17083

    audit_df = audit_duration_cap(
        df=df,
        model_name=model_name,
        cohort_name="Full_Test_Cohort",
        duration_cap=DURATION_CAP
    )

    full_audit_results.append(
        audit_df
    )

full_cap_audit = pd.concat(
    full_audit_results,
    ignore_index=True
)

print("\nFULL TEST-COHORT CAP AUDIT")
print("=" * 90)

print(
    full_cap_audit[
        [
            "Model",
            "Lambda",
            "N_Capped",
            "Pct_Capped",
            "Max_Uncapped",
            "Total_Minutes_Removed"
        ]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# 5. FIXED 12-CASE COHORT CAP AUDIT
# ============================================================

fixed_audit_results = []

for model_name, path in MODEL_FILES.items():

    print(
        f"Auditing fixed 12-case cohort: {model_name}"
    )

    df = pd.read_csv(
        path
    )

    fixed_df = (
        df[
            df["LOG_ID"].isin(
                cohort_ids
            )
        ]
        .copy()
    )

    assert len(fixed_df) == 12, (
        f"{model_name}: expected 12 fixed cases, "
        f"found {len(fixed_df)}."
    )

    # Preserve the original fixed-cohort ordering
    fixed_df = (
        fixed_df
        .set_index("LOG_ID")
        .loc[cohort_ids]
        .reset_index()
    )

    audit_df = audit_duration_cap(
        df=fixed_df,
        model_name=model_name,
        cohort_name="Fixed_12_Case_Cohort",
        duration_cap=DURATION_CAP
    )

    fixed_audit_results.append(
        audit_df
    )

fixed_cap_audit = pd.concat(
    fixed_audit_results,
    ignore_index=True
)

print("\nFIXED 12-CASE COHORT CAP AUDIT")
print("=" * 90)

print(
    fixed_cap_audit[
        [
            "Model",
            "Lambda",
            "N_Capped",
            "Pct_Capped",
            "Max_Uncapped",
            "Total_Minutes_Removed"
        ]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# 6. MAIN REALISED-COMPARISON CAP IMPACT (LAMBDA = 0.5)
# ============================================================

lambda05_audit = (
    fixed_cap_audit[
        fixed_cap_audit["Lambda"] == 0.5
    ]
    [
        [
            "Model",
            "N_Capped",
            "Pct_Capped",
            "Max_Uncapped",
            "Total_Minutes_Removed",
            "Mean_Reduction_Capped_Cases",
            "Max_Reduction"
        ]
    ]
    .reset_index(drop=True)
)

print("\nCAP IMPACT FOR MAIN REALISED COMPARISON (lambda = 0.5)")
print("=" * 90)

print(
    lambda05_audit.to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# 7. COMPACT CAP-AUDIT SUMMARY
# ============================================================

key_lambdas = [
    0.0,
    0.5,
    1.0
]

compact_full = (
    full_cap_audit[
        full_cap_audit[
            "Lambda"
        ].isin(key_lambdas)
    ]
    [
        [
            "Model",
            "Lambda",
            "N_Capped",
            "Pct_Capped",
            "Max_Uncapped"
        ]
    ]
    .reset_index(drop=True)
)

print("\nCOMPACT FULL-COHORT CAP SUMMARY")
print("=" * 80)

print(
    compact_full.to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# 8. EXPORT AUDIT TABLES
# ============================================================

full_output = (
    "Results/"
    "duration_cap_audit_full_cohort.csv"
)

fixed_output = (
    "Results/"
    "duration_cap_audit_fixed_12_cases.csv"
)

lambda05_output = (
    "Results/"
    "duration_cap_audit_lambda05.csv"
)

compact_output = (
    "Results/"
    "duration_cap_audit_compact.csv"
)

full_cap_audit.to_csv(
    full_output,
    index=False
)

fixed_cap_audit.to_csv(
    fixed_output,
    index=False
)

lambda05_audit.to_csv(
    lambda05_output,
    index=False
)

compact_full.to_csv(
    compact_output,
    index=False
)

print("\nSaved:")
print(full_output)
print(fixed_output)
print(lambda05_output)
print(compact_output)